<a href="https://colab.research.google.com/github/VinayaSharada/KateelLearningDemosToStudents/blob/main/TreasuryAnalytics/O2CProcessMiningNotebook/o2c_process_mining_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# O2C Process Mining Notebook

This notebook uses the same fictional Asteron Order-to-Cash event log as the browser-based **O2C Process Mining Workbench**. The objective is not a tool tour. The objective is to show, in a transparent way, how finance can diagnose the actual process before deciding what to eliminate, standardize, enable, assure, and monitor.


## What we are trying to answer

1. What does the assumed happy path look like?
2. What variants did the event log actually reveal?
3. Where are lead time, queue time, and rework leaking attention?
4. Which cases cross a control boundary?
5. What should finance remove, standardize, enable, retain as human judgement, or monitor after launch?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW_CSV_URL = "https://raw.githubusercontent.com/VinayaSharada/KateelLearningDemosToStudents/main/TechUseCaseDemos/O2CProcessMiningWorkbench/Data/o2c_event_log.csv"
LOCAL_CSV_CANDIDATES = [
    Path("TechUseCaseDemos/O2CProcessMiningWorkbench/Data/o2c_event_log.csv"),
    Path("../TechUseCaseDemos/O2CProcessMiningWorkbench/Data/o2c_event_log.csv"),
    Path("../../TechUseCaseDemos/O2CProcessMiningWorkbench/Data/o2c_event_log.csv"),
]

def resolve_data_source():
    for candidate in LOCAL_CSV_CANDIDATES:
        if candidate.exists():
            return str(candidate)
    return RAW_CSV_URL

DATA_SOURCE = resolve_data_source()
TARGET_SLA_HOURS = 48
ACTIVE_TOUCH_HOURS = {
    "Order received": 0.3,
    "Credit approved": 1.2,
    "Credit review": 2.5,
    "Credit evidence completed": 0.6,
    "Goods delivered": 1.0,
    "Delivery dispute logged": 0.8,
    "Delivery dispute resolved": 1.6,
    "Invoice issued": 0.5,
    "Invoice reissued": 0.6,
    "Payment received": 0.2,
    "Short payment review": 2.2,
    "Deduction approved": 1.8,
    "Remittance chased": 1.0,
    "Cash applied": 0.4,
}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
print(f"Using data source: {DATA_SOURCE}")


### Why this setup cell matters

This cell defines the notebook inputs and the illustrative touch-time assumptions used later in the friction analysis. These touch times are **not audited facts**. They are fictional facilitator assumptions that help us separate active handling time from queue time.


In [ ]:
df = pd.read_csv(DATA_SOURCE, parse_dates=["timestamp"], keep_default_na=False)
df = df.sort_values(["case_id", "timestamp"]).reset_index(drop=True)
print(f"Rows: {len(df)}")
print(f"Cases: {df['case_id'].nunique()}")
df.head(10)


### How to interpret the raw event log

Each row is one event in one O2C case. The event log is the source of truth for the demo. If a metric or variant is not traceable back to these rows, we should not trust it.


In [ ]:
happy_path = [
    "Order received",
    "Credit approved",
    "Goods delivered",
    "Invoice issued",
    "Payment received",
    "Cash applied",
]
print("Assumed happy path:")
print(" -> ".join(happy_path))
print(f"Assumed target SLA: {TARGET_SLA_HOURS} hours ({TARGET_SLA_HOURS/24:.1f} business days, illustrative)")


### Why we start with the happy path

The happy path is not the answer. It is the baseline assumption that many transformation discussions start with. The whole point of process mining is to test whether the executed path behaves like that assumption.


In [ ]:
cases = []
for case_id, group in df.groupby("case_id"):
    group = group.sort_values("timestamp").reset_index(drop=True)
    sequence = group["activity"].tolist()
    first = group.iloc[0]
    last = group.iloc[-1]
    lead_hours = (last["timestamp"] - first["timestamp"]).total_seconds() / 3600
    touch_hours = sum(ACTIVE_TOUCH_HOURS.get(activity, 0.5) for activity in sequence)
    wait_hours = max(0, lead_hours - touch_hours)
    repeated_count = len(sequence) - len(set(sequence))
    exception_rows = group[group["exception_type"] != "None"]
    exception_type = exception_rows.iloc[0]["exception_type"] if not exception_rows.empty else "None"
    exception_age_hours = ((last["timestamp"] - exception_rows.iloc[0]["timestamp"]).total_seconds() / 3600) if not exception_rows.empty else 0
    variant = " -> ".join(sequence)
    first_pass = exception_type == "None" and repeated_count == 0
    stp = variant == " -> ".join(happy_path)
    control_violation = ("Cash applied" in sequence and "Credit evidence completed" in sequence and sequence.index("Cash applied") < sequence.index("Credit evidence completed"))
    cases.append({
        "case_id": case_id,
        "entity": first["entity"],
        "amount_inr": first["amount_inr"],
        "variant": variant,
        "lead_hours": lead_hours,
        "touch_hours": touch_hours,
        "wait_hours": wait_hours,
        "rework_loops": repeated_count,
        "handoffs": len(sequence) - 1,
        "exception_type": exception_type,
        "exception_age_hours": exception_age_hours,
        "status": last["invoice_status"],
        "current_owner": last["owner"],
        "first_pass": first_pass,
        "stp": stp,
        "control_violation": control_violation,
    })

case_df = pd.DataFrame(cases)
case_df.head()


### What this case-level table gives us

This table converts event rows into case-level O2C metrics. The most important shift is conceptual: now we can compare the designed process to the executed process using lead time, wait time, rework, and control status at the case level.


In [ ]:
variant_summary = (
    case_df.groupby("variant", as_index=False)
    .agg(
        cases=("case_id", "count"),
        avg_lead_hours=("lead_hours", "mean"),
        avg_wait_hours=("wait_hours", "mean"),
        avg_rework_loops=("rework_loops", "mean"),
    )
    .sort_values("cases", ascending=False)
)
variant_summary.head(8)


### How to read the variants

A process variant is a unique ordered sequence of activities. If the happy path is the dominant variant, the process is comparatively stable. If many variants carry meaningful volume, then the designed process is hiding operational complexity.


In [ ]:
top_variants = variant_summary.head(6).copy()
plt.figure(figsize=(12, 5))
plt.barh(top_variants["variant"].str.slice(0, 70), top_variants["cases"], color="#0f766e")
plt.gca().invert_yaxis()
plt.xlabel("Cases")
plt.title("Top O2C Variants")
plt.tight_layout()
plt.show()


### What the chart should tell students

The chart should visibly separate the clean path from the exception paths. The key insight is not that the process is messy. The key insight is **which kinds of mess dominate**, and whether those patterns are stable enough for workflow enablement or still too dependent on human judgement and upstream discipline.


In [ ]:
summary = pd.DataFrame({
    "metric": [
        "Cases",
        "Variant count",
        "Average lead time (hours)",
        "Average wait time (hours)",
        "Rework rate",
        "First-pass yield",
        "Straight-through-processing rate",
    ],
    "value": [
        case_df["case_id"].nunique(),
        case_df["variant"].nunique(),
        round(case_df["lead_hours"].mean(), 1),
        round(case_df["wait_hours"].mean(), 1),
        f"{(case_df['rework_loops'].gt(0).mean()*100):.1f}%",
        f"{(case_df['first_pass'].mean()*100):.1f}%",
        f"{(case_df['stp'].mean()*100):.1f}%",
    ]
})
summary


### Why these metrics matter to finance

- **Lead time** shows how long it takes to turn an order into cash application.
- **Wait time** reveals the part of the timeline that is usually easier to redesign than touch time.
- **Rework rate** shows how often the process loops back.
- **First-pass yield** and **STP rate** help distinguish a stable path from a path that still depends on exceptions.


In [ ]:
friction_by_exception = (
    case_df.groupby("exception_type", as_index=False)
    .agg(
        cases=("case_id", "count"),
        avg_lead_hours=("lead_hours", "mean"),
        avg_wait_hours=("wait_hours", "mean"),
        avg_handoffs=("handoffs", "mean"),
        avg_rework_loops=("rework_loops", "mean"),
    )
    .sort_values("avg_wait_hours", ascending=False)
)
friction_by_exception


### How to interpret friction by exception

This is where the discussion becomes managerial rather than technical. If wait time is consistently large in a given exception type, the question is not “Which tool should we buy?” The question is “Why does work wait there, and is that delay a policy, discipline, evidence, or ownership issue?”


In [ ]:
breaches = []
for _, row in case_df.iterrows():
    breach_list = []
    if row["control_violation"]:
        breach_list.append("Cash applied before approval evidence completed")
    if row["exception_type"] == "Short payment" and row["exception_age_hours"] > 24:
        breach_list.append("Short payment unresolved beyond SLA")
    if row["exception_type"] != "None" and row["amount_inr"] > 500000 and row["wait_hours"] > 4:
        breach_list.append("High-value exception waiting beyond assignment window")
    for breach in breach_list:
        breaches.append({
            "case_id": row["case_id"],
            "amount_inr": row["amount_inr"],
            "breach": breach,
            "age_hours": round(max(row["exception_age_hours"], row["wait_hours"]), 1),
            "current_owner": row["current_owner"],
            "evidence_status": "Completed after the fact" if row["control_violation"] else "Needs review",
        })

breach_df = pd.DataFrame(breaches).sort_values(["amount_inr", "age_hours"], ascending=[False, False])
breach_df.head(10)


### Why the breach list matters

A control exception is not just another delay bucket. It crosses a governance boundary. The role of process mining here is to surface the behaviour clearly; the role of management is to decide what must remain controlled and human-reviewed.


In [ ]:
decision_board = pd.DataFrame([
    {
        "mined_insight": "Repeated credit review delays release",
        "decision_category": "Eliminate / standardize",
        "example_action": "Tighten credit evidence and master-data discipline before workflow enablement.",
        "expected_metric": "Credit-hold cycle time",
        "owner": "Credit control lead",
        "proof_point_30d": "20% lower credit-hold turnaround"
    },
    {
        "mined_insight": "Delivery disputes create long queue time",
        "decision_category": "Redesign upstream process",
        "example_action": "Improve proof-of-delivery and receiving discipline before automating downstream routing.",
        "expected_metric": "Dispute ageing",
        "owner": "Operations owner",
        "proof_point_30d": "Median dispute ageing below 24h"
    },
    {
        "mined_insight": "Stable standard cases can be routed automatically",
        "decision_category": "Enable workflow",
        "example_action": "Use low-code routing, reminders, and escalations only on the stable path.",
        "expected_metric": "Straight-through-processing rate",
        "owner": "O2C process owner",
        "proof_point_30d": "STP above 60% for four weeks"
    },
    {
        "mined_insight": "Short-payment and deduction review still need judgement",
        "decision_category": "Keep human-reviewed",
        "example_action": "Provide evidence workbench and controller ownership instead of trying to automate away judgement.",
        "expected_metric": "Short-payment queue ageing",
        "owner": "Collections controller",
        "proof_point_30d": "Aged short-payment cases reduced by half"
    },
    {
        "mined_insight": "Control breaches need visible monitoring",
        "decision_category": "Assure / monitor",
        "example_action": "Log trigger, owner, evidence, and escalation deadline whenever the process crosses a control line.",
        "expected_metric": "Post-facto evidence completion",
        "owner": "Controller + internal controls",
        "proof_point_30d": "Zero new cash-applied-before-evidence cases"
    },
])
decision_board


### How to use the decision board

This is the bridge from diagnosis to action. A mined insight is not the action itself. The decision board forces us to choose what should be eliminated, standardized, enabled, retained under judgement, or monitored.


In [ ]:
monitoring = pd.DataFrame([
    {
        "trigger": "High-value exception unassigned > 4h",
        "hit": bool(((case_df["amount_inr"] > 500000) & (case_df["exception_type"] != "None") & (case_df["wait_hours"] > 4)).any()),
        "owner": "O2C controller and sales owner",
        "evidence_logged": "Case owner, status, and ageing",
        "deadline": "4-hour escalation"
    },
    {
        "trigger": "Delivery dispute unresolved > 24h",
        "hit": bool(((case_df["exception_type"] == "Delivery dispute") & (case_df["exception_age_hours"] > 24)).any()),
        "owner": "Operations owner",
        "evidence_logged": "Dispute reason and POD note",
        "deadline": "24-hour escalation"
    },
    {
        "trigger": "Short-payment queue above five cases",
        "hit": bool((case_df["exception_type"] == "Short payment").sum() > 5),
        "owner": "Collections controller",
        "evidence_logged": "Deduction queue with reviewer",
        "deadline": "Same-day review"
    },
    {
        "trigger": "STP rate below 60%",
        "hit": bool(case_df["stp"].mean() < 0.60),
        "owner": "O2C process owner",
        "evidence_logged": "Weekly STP trend",
        "deadline": "Weekly process review"
    },
])
monitoring


### Why monitoring is part of transformation

Process mining is not a one-off diagnostic. If the organization does not define triggers, owners, and escalation deadlines, the process will drift back into the same loops even after a redesign project.


## What a strong classroom conclusion sounds like

A strong conclusion does **not** say: “The answer is automation.”

A strong conclusion says something like:

- the stable path is clear enough for workflow enablement,
- credit-hold and dispute issues need upstream discipline first,
- short-payment review still requires human judgement,
- and control breaches need visible monitoring after launch.
